# Agent Patterns

**Prerequisites:** notebooks 2 through 5 of this chapter

Agentic patterns are usually taught as a list of architectures — ReAct, Plan & Execute, ReWOO, Tree of Thoughts, Reflexion, Debate — each with its own diagram, as though each needed its own system.

They don't. Every one of them is a different arrangement of the same handful of components, and this notebook writes each as a declared loop over what notebooks 2 through 5 already built. Nothing new gets implemented here. That's the point: if a pattern needs new machinery, the decomposition was wrong.

**By the end you'll have:**
- Eleven named patterns written as declared loops, each one a list of components already in `runtime.py`, `observe.py`, `think.py`, and `act.py`
- A clear view of what actually distinguishes them — usually one component, or one flag
- Every loop constructed for real, so none of it is pseudocode
- An honest account of the one thing these components genuinely can't express

## Setup

Every import is something built earlier in this chapter. Nothing is defined in this notebook.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))

from runtime import Runtime, Loop, Graph
from observe import Observe
from think import Think, PlannerThink, ReflectionThink, SelectThink, plan_done
from act import Act

In [2]:
# Stand-ins so the declarations below are real objects. A concrete chapter would pass its own.
TOOLS, FUNCTIONS, SCHEMAS, OUTPUTS = [], {}, {}, {}
REQUEST = "the user's request"

def make_branch_runtime(branch):
    return Runtime(
        loop=[Think(tools=TOOLS, allow_fork=False,
                    temperature=branch.get("temperature"), top_p=branch.get("top_p")),
              Act(functions=FUNCTIONS, schemas=SCHEMAS),
              Observe()],
        repeat_from=0,
        max_iterations=branch.get("max_iterations") or 9,
    )

def describe(name, runtime):
    parts = []
    for c in runtime.loop:
        if isinstance(c, Loop):
            inner = " -> ".join(type(x).__name__ for x in c.components)
            stop = f"count={c.count}" if c.count is not None else f"until={c.until.__name__}"
            parts.append(f"Loop[{inner}, {stop}]")
        else:
            parts.append(type(c).__name__)
    tail = "" if runtime.repeat_from is None else f"   (repeat_from={runtime.repeat_from})"
    print(f"{name:18} {' -> '.join(parts)}{tail}")

PATTERNS = {}

## The primitives, one more time

Four kinds of thing, and everything below is built from them.

| | what it does | never does |
|---|---|---|
| **`Observe`** | brings something in — a user event, or a raw effect turned into a readable observation | decide what happens next |
| **`Think`** | reasons over the conversation and produces a decision, plan, or critique | change anything outside the program |
| **`Act`** | carries out a decision already made | decide anything |
| **`Runtime` / `Loop` / `Graph`** | schedule which component runs when | reason |

The Thinks differ only in what they're asked and what shape they return: `Think` decides a tool call, a delegation, or an answer; `PlannerThink` writes a whole sequence; `ReflectionThink` critiques without answering; `SelectThink` scores candidates and prunes.

## Chain of Thought

Reason through a problem, then answer. No tools, no cycle.

```
Observe(event) -> Think -> Act
```

`Think` with no tools has nothing to call, so its only available decision is `respond`. `Act` turns that into the `assistant` message and `Runtime` stops. There's no `repeat_from`, so it walks the list once — by notebook 1's definition this is a workflow, not an agent, and that's the correct classification. Nothing about it adapts.

In [3]:
PATTERNS["Chain of Thought"] = Runtime(loop=[
    Observe(event=REQUEST),
    Think(allow_fork=False),
    Act(),
])

## ReAct

Interleave reasoning and acting: decide one step, do it, look at the result, decide again.

```
Observe(event) -> [ Think -> Act -> Observe ]  repeating
```

The one structural difference from Chain of Thought is `repeat_from=1`, plus tools for `Think` to reach for. Notebook 2 built this.

In [4]:
PATTERNS["ReAct"] = Runtime(
    loop=[
        Observe(event=REQUEST),
        Think(tools=TOOLS, allow_fork=False),
        Act(functions=FUNCTIONS, schemas=SCHEMAS),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=20,
)

## Plan & Execute

Decide the whole sequence first, then carry it out without re-deciding at every step.

```
Observe(event) -> PlannerThink -> Loop[ Act -> Observe, until=plan_done ]  repeating
```

`Think` is replaced by `PlannerThink`, and the execution phase has no Think in it at all — `Observe` emits each next step from the plan. Cycling back to `PlannerThink` after `plan_done` is what makes replanning possible: its second job, `ContinueOrFinish`, either answers or appends more steps. Notebook 4 built this.

In [5]:
PATTERNS["Plan & Execute"] = Runtime(
    loop=[
        Observe(event=REQUEST),
        PlannerThink(tools=TOOLS, outputs=OUTPUTS),
        Loop([Act(functions=FUNCTIONS, schemas=SCHEMAS), Observe()], until=plan_done),
    ],
    repeat_from=1,
)

## ReWOO

*Reasoning WithOut Observation.* The planner writes every step up front using variable references, workers execute them, and a solver composes the answer — with the model never consulted between steps.

```
Observe(event) -> PlannerThink -> Graph -> Think -> Act
```

What defines ReWOO is that the plan carries **variable references** so no intermediate result has to go back to the model to decide the next call. That's exactly what `$stepN` is. Notebook 4 introduced it to solve the unknown-argument problem; ReWOO is the pattern named after treating it as the whole execution strategy.

`Graph` here rather than `Loop`, because once nothing needs re-deciding there's no reason to run independent steps one at a time. The trailing `Think` is ReWOO's solver.

In [6]:
PATTERNS["ReWOO"] = Runtime(loop=[
    Observe(event=REQUEST),
    PlannerThink(tools=TOOLS, outputs=OUTPUTS),
    Graph(act=Act(functions=FUNCTIONS, schemas=SCHEMAS)),
    Think(allow_fork=False),
    Act(),
])

## LLM Compiler

Plan into a DAG, execute in parallel, and replan when the results warrant it.

```
Observe(event) -> PlannerThink -> Graph  repeating
```

Structurally almost ReWOO, with one difference that matters: `repeat_from=1` sends control back to `PlannerThink`, whose `ContinueOrFinish` can add more steps rather than being forced to answer from what the first plan produced. ReWOO commits; the compiler pattern re-enters.

The parallel scheduling both rely on is `Graph`'s: a step starts the moment its own dependencies are satisfied, which notebook 4 measured.

In [7]:
PATTERNS["LLM Compiler"] = Runtime(
    loop=[
        Observe(event=REQUEST),
        PlannerThink(tools=TOOLS, outputs=OUTPUTS),
        Graph(act=Act(functions=FUNCTIONS, schemas=SCHEMAS)),
    ],
    repeat_from=1,
    max_iterations=20,
)

## Self-Refine

Produce an answer, critique it, revise. One agent, no tools.

```
Observe(event) -> Think -> Act -> ReflectionThink  repeating
```

`ReflectionThink` never writes the answer (notebook 3), so the revision comes from the `Think` at the top of the next pass reading the critique as context. When that `Think` judges the work good enough it responds instead, `Act` emits the `assistant` message, and the run ends. No `good_enough` branch is wired anywhere — the field is information for a `Think`, not a control-flow signal.

In [8]:
PATTERNS["Self-Refine"] = Runtime(
    loop=[
        Observe(event=REQUEST),
        Think(allow_fork=False),
        Act(),
        ReflectionThink(),
    ],
    repeat_from=1,
    max_iterations=12,
)

## Reflexion

Self-Refine with tool use: attempt the task over several turns, then critique the whole attempt rather than a single answer.

```
Observe(event) -> Loop[ Think -> Act -> Observe, count=3 ] -> ReflectionThink  repeating
```

`Loop(count=3)` is what makes the critique land on a complete attempt instead of one turn — the difference between "that tool call was wrong" and "that approach was wrong." `repeat_from=1` re-enters the same `Loop`, so the next attempt is made with the critique in hand. Notebook 3 ran exactly this.

In [9]:
PATTERNS["Reflexion"] = Runtime(
    loop=[
        Observe(event=REQUEST),
        Loop([Think(tools=TOOLS, allow_fork=False), Act(functions=FUNCTIONS, schemas=SCHEMAS), Observe()], count=3),
        ReflectionThink(),
    ],
    repeat_from=1,
    max_iterations=24,
)

## Self-Consistency

Answer the same question several times independently, then reconcile.

```
Observe(event) -> Think(fork) -> Act -> Observe -> Think -> Act
```

The fork machinery from notebook 5, with every branch given the *same* goal and a different `temperature`/`top_p` — chapter 1's sampling knobs used to generate disagreement rather than suppress it. `Observe` merges the branch results into one labelled observation and the trailing `Think` reconciles them.

There is no voting component, deliberately. A `Think` reading three labelled answers can weigh them however the situation deserves; a `VoteThink` would hardcode majority.

In [10]:
PATTERNS["Self-Consistency"] = Runtime(loop=[
    Observe(event=REQUEST),
    Think(allow_fork=True),
    Act(branch_runtime=make_branch_runtime),
    Observe(),
    Think(allow_fork=False),
    Act(),
])

## Debate

Several agents argue positions, see each other's turns, and keep going.

```
Observe(event) -> Think(fork) -> Act -> Observe
```

The same four components as Self-Consistency. The entire difference is in the fork decision: `persistent=True` keeps each branch's conversation alive on `conversation.branches`, and `broadcast=True` copies each branch's turn into its siblings' histories, attributed. Opposing positions come from the goals the delegating `Think` wrote.

`advance_branch` gives a branch another turn. `Act` implements it; nothing yet decides *when* to use it or *whose* turn is next — with more than two debaters that's a real judgment call, and notebook 5 issues it by hand rather than baking a round-robin policy into `Act`.

In [11]:
PATTERNS["Debate"] = Runtime(loop=[
    Observe(event=REQUEST),
    Think(allow_fork=True),
    Act(branch_runtime=make_branch_runtime),
    Observe(),
])

## Tree of Thoughts

Generate several candidate directions, evaluate them, keep the best few, and expand those.

```
Observe(event) -> Think(fork) -> Act -> SelectThink(keep=2) -> Act -> Observe  repeating
```

This is beam search, and every piece already existed. `Think` forks *k* candidates; `SelectThink` scores them while they're still raw in `pending` and emits a `prune`; `Act` applies the index filter; `Observe` merges only the survivors; `repeat_from=1` returns to `Think`, which now forks from the frontier rather than from scratch. Notebook 5 ran it.

The pruned candidates never reach `messages`, so the context holds the frontier rather than the whole tree.

In [12]:
PATTERNS["Tree of Thoughts"] = Runtime(
    loop=[
        Observe(event=REQUEST),
        Think(allow_fork=True),
        Act(branch_runtime=make_branch_runtime),
        SelectThink(keep=2),
        Act(),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=24,
)

## Graph of Thoughts

Tree of Thoughts plus two operations a tree doesn't have: **aggregating** several thoughts into one, and **refining** a thought in place.

```
Observe(event) -> Think(fork, persistent) -> Act -> SelectThink(keep=3) -> Act -> Observe  repeating
```

The declaration is Tree of Thoughts with `persistent=True` on the fork, and both extra operations fall out of what that enables:

- **Aggregation** — `Observe` merges the surviving branches into one labelled observation, so the next `Think` sees several thoughts at once and can fork a new branch whose goal is built from all of them. Merging several results into one input is the aggregation edge.
- **Refinement** — `advance_branch` continues a persistent branch with its own history intact, rather than starting a fresh branch from a restated goal. That's a thought being improved rather than replaced.

The declared loop is the same; what makes it a graph rather than a tree is that surviving branches stay addressable, so an edge can point back at one.

In [13]:
PATTERNS["Graph of Thoughts"] = Runtime(
    loop=[
        Observe(event=REQUEST),
        Think(allow_fork=True),
        Act(branch_runtime=make_branch_runtime),
        SelectThink(keep=3),
        Act(),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=30,
)

## All eleven

Every loop above is a real object, constructed from real classes. Printing their component sequences is the check that none of this is pseudocode — a name that didn't exist, or an argument that wasn't accepted, would have raised on construction:

In [14]:
for name, runtime in PATTERNS.items():
    describe(name, runtime)

Chain of Thought   Observe -> Think -> Act
ReAct              Observe -> Think -> Act -> Observe   (repeat_from=1)
Plan & Execute     Observe -> PlannerThink -> Loop[Act -> Observe, until=plan_done]   (repeat_from=1)
ReWOO              Observe -> PlannerThink -> Graph -> Think -> Act
LLM Compiler       Observe -> PlannerThink -> Graph   (repeat_from=1)
Self-Refine        Observe -> Think -> Act -> ReflectionThink   (repeat_from=1)
Reflexion          Observe -> Loop[Think -> Act -> Observe, count=3] -> ReflectionThink   (repeat_from=1)
Self-Consistency   Observe -> Think -> Act -> Observe -> Think -> Act
Debate             Observe -> Think -> Act -> Observe
Tree of Thoughts   Observe -> Think -> Act -> SelectThink -> Act -> Observe   (repeat_from=1)
Graph of Thoughts  Observe -> Think -> Act -> SelectThink -> Act -> Observe   (repeat_from=1)


Read down that list and the family resemblances are hard to miss.

**ReAct and Chain of Thought** differ by `repeat_from` alone. That single index is the whole distance between a workflow and an agent.

**Plan & Execute, ReWOO, and LLM Compiler** all start `Observe -> PlannerThink`. They differ in how the plan is executed (`Loop` one at a time vs. `Graph` by dependency) and whether control returns to the planner. Three "different architectures", one planner, two schedulers.

**Self-Refine and Reflexion** are the same shape; Reflexion wraps the attempt in a `Loop` so the critique lands on a whole attempt rather than one turn.

**Self-Consistency, Debate, Tree of Thoughts, and Graph of Thoughts** all fork. Self-Consistency and Debate share an identical component list and differ only in the `persistent` and `broadcast` flags on the decision — which means the difference between them isn't in the code at all; it's in what a `Think` decided at runtime. Tree of Thoughts adds `SelectThink`, and Graph of Thoughts adds persistence to that.

The patterns aren't eleven architectures. They're a few real choices: whether to plan ahead, whether to loop, whether to split, whether the splits persist and can see each other, and whether the results get filtered.

## What this can't do

Two honest limits.

**Backtracking to a discarded candidate.** Once `SelectThink` prunes, the losing candidates are dropped from `pending` and never enter `messages`. Classic Tree of Thoughts can abandon a line, exhaust the alternatives, and come back to it. This can't — a pruned branch isn't addressable anymore. Keeping every candidate scored on the conversation and letting a `Think` reopen one would fix it, and it's bookkeeping rather than a structural change, but it isn't built.

**Deciding whose turn is next.** `advance_branch` works, but no `Think` produces one. In a two-sided debate alternation is obvious; past that, choosing which branch speaks next is judgment, and the reason it's unbuilt is that the obvious implementation — round-robin inside `Act` — would put a policy decision in the component that isn't allowed to make them.

Neither is a limit of the decomposition. Both are work that hasn't been done.

## What you built

Nothing. That's the result.

✓ Wrote eleven named patterns as declared loops over `Runtime`, `Loop`, `Graph`, `Observe`, the four Thinks, and `Act` — with no component added, and no `runtime.py` change between any two of them

✓ Constructed every one for real, so the declarations are verified code rather than diagrams

✓ Located what actually separates them: `repeat_from` for Chain of Thought vs. ReAct; `Loop` vs. `Graph` for Plan & Execute vs. ReWOO; two boolean flags — decided at runtime, not written in the loop — for Self-Consistency vs. Debate

✓ Named the two things that genuinely aren't built: backtracking to a pruned candidate, and deciding which persistent branch speaks next

This closes `02_agent_runtime`. The chapter began with one fused function from chapter 1 that could run exactly one agent, and ends with eleven patterns that are all the same runtime walking a different list.